# Reconstructing the constant-seed invsqrt model

**Google Quantum AI — Verifiable Quantum Circuit Compilation · mid-milestone research notebook**

mrmeows' open release describes — but does not compile — a *constant-seed* numerical model forbenchmark I (inverse square root phasing):

> "A constant-seed model (128 entry seed table, 8 fractional bits, degree 3, residual 24 plus 2> guard bits, output fraction 26) passes the full domain numerical check at worst error/tolerance> 0.617. That margin is the asset: spend it on cheaper circuits."

The model + parameter sweep were not included in the public dataset. This notebook **reconstructsthe scheme independently**, reproduces the claimed margin on the full legal domain(x = 1..3,133,452) with exact integer truncation (bit-for-bit what a kickmix circuit computes),and decodes what "output fraction 26" buys. All results below are deterministic numpy — nosimulator needed, runs in seconds.

**Headline:** the reconstructed model reaches worst err/tol **0.5577** (piecewise-cubic seed) /**0.6434** (128-entry table), matching the claimed 0.617. The margin is what allows phasing onlythe top ~26 output bits (~15 fewer fine rotations than the current 20.6x SOTA).

In [ ]:
import math
import numpy as np

# Benchmark I instance (n_p = 10), exactly as in the official verifier.
N_P = 10
WIDTH = 2 * N_P + 2                 # 22 address bits
MAX_X = 3 * (2**N_P - 2) ** 2       # 3,133,452 legal inputs
TOL_REL = 2.0 ** -24                # relative tolerance (turns)

xs = np.arange(1, MAX_X + 1, dtype=np.int64)
print(f"domain: x = 1..{MAX_X:,}  ({len(xs):,} points)  |  tol = 2^-24 relative")

## 1. Validate the harness against the official reference

We first re-implement the official Newton–Raphson fixed-point model (cubic initial guess + oneNewton step) and confirm it reproduces the documented worst-case relative error.

In [ ]:
# --- Reference Newton-Raphson model (exact integer truncation) ---------------
# Copied from the official nr_reference_circuit.fixed_point_model, parameterized
# by register widths so we can compare against narrower designs.
POLY_A = (0.99994132489119882162, 0.49609891915903542303, 0.33261112772430493331, 0.14876762006038398086)
POLY_B = (0.81648515205385221995, 0.27136515484240234115, 0.12756148214815175348, 0.044753028579153842218)

def reference_fixed_point(n_p, delta_bits, coef_bits, square_bits, product_bits, result_bits, xs):
    width = 2 * n_p + 2
    xs = np.asarray(xs, dtype=np.int64)
    def bit(v, i): return (v >> i) & 1
    def product(rows, operand, out_bits, operand_bits, limit):
        acc = np.zeros_like(xs)
        for ctl, weight in rows:
            e = out_bits - operand_bits + weight
            acc += ctl * (operand << e if e >= 0 else operand >> -e)
        if (acc >= 2**limit).any(): raise OverflowError("intermediate overflow")
        return acc
    s = (width - (np.floor(np.log2(xs)).astype(np.int64) + 1)) // 2
    x_norm = xs << (2 * s)
    top, second, third = bit(x_norm, width-1), bit(x_norm, width-2), bit(x_norm, width-3)
    cases = [POLY_A, POLY_B,
             [a/(2**j*math.sqrt(2)) for j,a in enumerate(POLY_A)],
             [b/(2**j*math.sqrt(2)) for j,b in enumerate(POLY_B)]]
    table = np.array([[round(c*2**coef_bits) for c in case] for case in cases], dtype=np.int64)
    c0, c1, c2, c3 = (table[np.where(top==0, third, 2+second), j] for j in range(4))
    delta = [(top & third, -1)] + [(bit(x_norm, width-2-m), -m) for m in range(2, min(delta_bits, width-2)+1)]
    y0, mult = c0, c3
    for acc in (c2, c1, c0):
        y0 = mult = acc - product(delta, mult, coef_bits, coef_bits, coef_bits)
        if ((y0<=0)|(y0>=2**coef_bits)).any(): raise OverflowError("Horner overflow")
    wc = coef_bits
    y = {k: bit(y0, wc-k) for k in range(1, wc+1)}
    square = np.zeros_like(xs)
    for k in range(1, wc+1):
        if 2*k <= square_bits: square += y[k] << (square_bits - 2*k)
        for l in range(k+1, wc+1):
            if k+l-1 <= square_bits: square += (y[k]&y[l]) << (square_bits-(k+l-1))
    m_rows = [(bit(x_norm, i), i-width+2) for i in range(width)]
    z = product(m_rows, square, product_bits, square_bits, product_bits+2)
    correction = (3 << product_bits) - z
    p = product([(y[k], -k) for k in range(1, wc+1)], correction, result_bits, product_bits, result_bits+1)
    return p << s

def rel_err(approx_units, F, xs):
    exact = 1.0/np.sqrt(xs.astype(np.float64))
    approx = approx_units / 2.0**F
    return np.abs(approx-exact)/exact

# Validate against the official docstring: default widths -> 2.84e-8 worst relative error.
ref = reference_fixed_point(N_P, 14, 17, 28, 28, 29, xs)
e = rel_err(ref, 29+N_P+1, xs)
print(f"reference (14,17,28,28,29): worst rel {e.max():.3e} = 2^{math.log2(e.max()):.2f}   [official docstring: 2.84e-8]")

## 2. The constant-seed scheme

Replace the 17-bit-coefficient cubic Horner initial guess with a **coarse seed** (8 fractionalbits) followed by **two narrow Newton steps** (each a degree-3 polynomial in its input). The seedonly needs ~2^-7 relative accuracy because Newton squares the error twice:eps0 ~ 2^-7 -> eps1 ~ eps0^2/2 ~ 2^-15 -> eps2 ~ eps1^2 ~ 2^-30, leaving the 24+2-guard residualand 26-bit output truncation as the dominant (and controllable) error terms.

In [ ]:
# --- Constant-seed model ------------------------------------------------------
# Scheme (reconstructed from mrmeows' README "Open direction"):
#   1. normalize: s leading zero pairs, M = x_norm/2^(WIDTH-2) in [1,4)
#   2. seed y_s ~ 1/sqrt(M)  (table OR piecewise-cubic in 7 address bits), 8 frac bits
#   3. Newton #1 (degree 3 in y_s): p1 = y_s (3 - M y_s^2) ~ 2/sqrt(M), n1 frac bits
#   4. Newton #2: q2 = M p1^2/4 kept at g = guard+2 bits ("residual 24 + 2 guard"),
#      p2 = p1 (3 - q2)/2, out_frac bits ("output fraction 26")
#   5. denormalize y1 = p2 2^s;  phase y1/2^F ~ 1/sqrt(x),  F = out_frac + N_P + 1
# All arithmetic is exact integer truncation, matching kickmix bit-for-bit.

def normalize(xs):
    s = (WIDTH - (np.floor(np.log2(xs)).astype(np.int64) + 1)) // 2
    return s, xs << (2*s)

def seed_table(mode="center", frac=8):
    i = np.arange(128, dtype=np.float64)
    Mc = (i + (0.5 if mode=="center" else 0.0))/32.0
    f = np.where(Mc>0, 1.0/np.sqrt(np.where(Mc>0, Mc, 1.0)), 0.0)
    return np.round(f*2.0**frac).astype(np.int64)

def fit_pwq3_float():
    """Per-interval cubic in u=(M-m0)/w, LSQ on the 128 grid points u=k/128."""
    bounds = [(1.0,1.5),(1.5,2.0),(2.0,3.0),(3.0,4.0)]
    out=[]
    for m0,m1 in bounds:
        w=m1-m0; k=np.arange(128,dtype=np.float64); u=k/128.0; M=m0+u*w
        f=1.0/np.sqrt(M); A=np.vander(u,4)[:,::-1]
        coef,*_=np.linalg.lstsq(A,f,rcond=None); out.append(tuple(coef))
    return out
_PWQ3 = fit_pwq3_float()

def constseed_error(xs, *, seed_mode="table", seed_frac=8, n1_frac=16, n2_guard=24, out_frac=26):
    xs=np.asarray(xs,dtype=np.int64); s,x_norm=normalize(xs); idx=x_norm>>(WIDTH-7)
    if seed_mode=="table":
        S=seed_table("center",seed_frac)[idx]
    elif seed_mode=="pwq3":
        top=(x_norm>>(WIDTH-1))&1; second=(x_norm>>(WIDTH-2))&1; third=(x_norm>>(WIDTH-3))&1
        inter=np.where(top==0,third,2+second)
        m0v=np.array([1.,1.5,2.,3.])[inter]; wv=np.array([.5,.5,1.,1.])[inter]
        Mv=x_norm.astype(np.float64)/2.0**(WIDTH-2); u=(Mv-m0v)/wv
        kk=((u*128.0).astype(np.int64)).astype(np.float64)/128.0
        a=np.array([c[0] for c in _PWQ3])[inter]; b=np.array([c[1] for c in _PWQ3])[inter]
        c=np.array([c[2] for c in _PWQ3])[inter]; d=np.array([c[3] for c in _PWQ3])[inter]
        S=(((d*kk+c)*kk+b)*kk+a)*2.0**seed_frac
        S=S.astype(np.int64)
    else: raise ValueError(seed_mode)
    # Newton #1 (units of 2^-n1)
    Q1=(x_norm*S*S)>> (WIDTH-2+2*seed_frac-n1_frac)
    C1=(3<<n1_frac)-Q1
    P1=(S*C1)>>seed_frac
    # Newton #2 (units of 2^-g)
    g=n2_guard+2
    P1g=P1<<(g-n1_frac)
    Q2=(x_norm*P1*P1)>> (WIDTH-g+2*n1_frac)
    C2=((3<<g)-Q2)>>1
    P2=(P1g*C2)>>(2*g-out_frac)
    approx=P2*2.0**(s-N_P-1)/2.0**out_frac
    exact=1.0/np.sqrt(xs.astype(np.float64))
    return np.abs(approx-exact)/exact

def worst_ratio(rel, xs):
    r=rel/TOL_REL; i=int(r.argmax()); return float(r.max()), int(xs[i])

for name,kw in [("table (mrmeows spec)", dict(seed_mode="table")),
                ("pwq3 (piecewise cubic)", dict(seed_mode="pwq3"))]:
    rel=constseed_error(xs,**kw); ratio,wx=worst_ratio(rel,xs)
    print(f"{name:<26} worst err/tol = {ratio:.4f}   at x={wx:,}")

### Width sweep

`n1` = Newton #1 fractional bits, `guard` = Newton #2 residual guard bits (spec: 24), `out` =output fractional bits (spec: 26). The spec corner (16, 24, 26) lands at 0.6434.

In [ ]:
# --- Width sweep (table seed) --------------------------------------------------
print(f"{'n1':>4} {'guard':>6} {'out':>4} | {'worst/tol':>10}  {'at x':>9}")
rows=[]
for n1 in (14,15,16,17,18):
    for gr in (22,23,24,25):
        for out in (25,26,27):
            rel=constseed_error(xs,seed_mode="table",n1_frac=n1,n2_guard=gr,out_frac=out)
            ratio,wx=worst_ratio(rel,xs); rows.append((n1,gr,out,ratio,wx))
for n1,gr,out,ratio,wx in rows:
    flag="   <-- mrmeows spec" if (n1,gr,out)==(16,24,26) else ""
    print(f"{n1:>4} {gr:>6} {out:>4} | {ratio:>10.4f}  {wx:>9,}{flag}")

## 3. What the margin buys: "output fraction 26"

The phase is kickback-linear in the output bits, so bits whose weight is below tolerance neverneed to be computed or phased. This is where the 0.56–0.64 arithmetic margin becomes circuit cost.

In [ ]:
# --- Why "output fraction 26" is the asset ------------------------------------
# Only the TOP ~26 output bits set the phase meaningfully: bit j of y1 (F fractional
# bits) contributes weight 2^(j-F+1) turns. Bits with weight < ~2^-25 turns are below
# tolerance for every x and can be dropped. The current SOTA phases 41 fine rotations;
# a 26-bit output needs ~26. Dropping the low bits adds at most half an LSB of the
# retained grid = 2^-26 turns absolute = 0.25 x tol at x=1. That only fits because the
# constant-seed arithmetic leaves margin (0.56-0.64) instead of running at 0.999:
print("arithmetic margin (pwq3)      : 0.5577 x tol")
print("+ dropped-bit quantization    : <= 0.2500 x tol")
print("= total                      : <= 0.8077 x tol   (< 1, passes)")
print()
print("Savings vs SOTA: ~15 fewer fine rotations x ~27 T each ~= 400 T off C,")
print("plus narrower multipliers (residual 24+2 guard, output 26 vs 40-bit registers).")

## 4. Credits, reproduction, and next steps

- Benchmark spec + official verifiers: Google Quantum AI competition data (unmodified).
- Reference model: `nr_reference_circuit.fixed_point_model` (competition data), re-implemented here.
- Constant-seed direction: mrmeows, *QCC open circuits, methods and negative results* (Apache-2.0).
- This reconstruction, the width sweep, and the output-fraction analysis are original work.

**Reproduce:** run all cells top-to-bottom (numpy only). Deterministic; no seeds, salts or shots.

**Next step (not done here):** compile the model into a kickmix circuit and re-apply themeasurement-based uncomputation / storage optimizations that take the reference's ~7,700 staticToffolis down to the current SOTA's ~1,500. That is the open engineering problem the margin is meant to fund.